# broadcasting-rules — ex8: per-channel bias add to a 4-D image batch

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcasting-rules`. Running the final beacon cell reports progress against the `Numpy: Vectorization and broadcasting` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcasting-rules`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Broadcasting — quick refresher

**The rule** (NumPy & PyTorch agree):
1. Right-align both shapes; left-pad the shorter with 1s.
2. For each pair of aligned axes: equal → keep; one is 1 → use the other; otherwise → incompatible.

**The dangerous case.** When a shape *almost* matches you can get an unintended broadcast that runs silently and produces wrong values. Always shape-check (`print(x.shape, y.shape, (x*y).shape)`) when wiring up a new pipeline.

### Exercise 8 — per-channel bias add to a 4-D image batch

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Add a per-channel bias vector to an NHWC image batch and visualize the per-channel shift.
> Keywords: multi-axis-broadcast, image-batch, channel-axis, heatmap, before-after
> ```

**KCs targeted:** `broadcast-leading-axes`, `per-channel-bias`, `axis-insertion-deep-tensor`

You have an image batch `X` of shape `(B, H, W, C)` and a per-channel bias `b` of shape `(C,)`. Implement `ex8_per_channel_bias(X, b)` to return `X + bias_broadcast` of shape `(B, H, W, C)` where `b[c]` is added to every spatial position of channel `c` of every image in the batch.

Don't loop. The whole job is one expression once `b` has the right shape.

**The shape-trace.** Right-align: `(B, H, W, C)` vs `(C,)` → `(C,)` gets left-padded to `(1, 1, 1, C)` and the bias broadcasts across `B`, `H`, `W`. That's the case NumPy/PyTorch handles for you automatically. Verify by adding a non-trivial bias `b = [1, 10, 100, 1000]` and check that channel 0 shifted by 1, channel 3 shifted by 1000.

The test cell then plots a `(H, W)` heatmap of channel 2 *before* and *after* the bias add so you can see the uniform offset — the spatial *pattern* is unchanged, only the level shifts.

In [ ]:
def ex8_per_channel_bias(X: Tensor, b: Tensor) -> Tensor:
    # b: (C,) right-aligns with last axis of X → automatic broadcast
    return X + b


<details><summary>Solution</summary>

```python
def ex8_per_channel_bias(X: Tensor, b: Tensor) -> Tensor:
    # b: (C,) right-aligns with last axis of X → automatic broadcast
    return X + b
```

**Why this is a 1-liner.** Right-align broadcasting was *designed* for this case: a 1-D parameter vector matching the trailing channel axis of a multi-D activation. No reshape needed.

**When does this break?** If your tensor is NCHW instead of NHWC, the channel axis is *not* the last axis. Then `X + b` broadcasts `b` over the wrong dimension (or fails). The fix is `X + b[:, None, None]` (insert two trailing axes so `b` becomes `(C, 1, 1)`, which right-aligns over `(N, C, H, W)`). This is one of the most common silent bugs in computer-vision code — see the next exercise.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()